# LeetCode 1603 — Design Parking System bằng Python

**Link đề:** https://leetcode.com/problems/design-parking-system/
**Độ khó:** Easy · **Chủ đề:** Design, Simulation, Counting

```python
class ParkingSystem:

    def __init__(self, big: int, medium: int, small: int):
        ...

    def addCar(self, carType: int) -> bool:
        ...
```

## Mục tiêu học
1. Làm quen bài **thiết kế (design)**: viết một **class** có trạng thái (state) và các method thay đổi trạng thái đó.
2. Hiểu `class`, `__init__`, `self`, **thuộc tính (attribute)** và **method** cho người mới.
3. Biết cách LeetCode gọi bài design (hai mảng: tên thao tác + tham số) và tự viết hàm chạy mô phỏng y hệt.
4. Viết 3 cách lưu trạng thái: 3 biến riêng + `if/elif` → **list đánh chỉ số theo carType** → `dict`; so sánh ưu nhược.
5. Trace trạng thái sau mỗi lệnh, test đối chiếu, mở rộng: rời bãi (`removeCar`), xe nhỏ được đỗ chỗ lớn hơn.

## 1. Đề bài

Thiết kế hệ thống bãi đỗ xe có 3 loại chỗ: **lớn (big)**, **vừa (medium)**, **nhỏ (small)**, mỗi loại có số chỗ cố định.

Cài đặt class `ParkingSystem`:
- `ParkingSystem(big, medium, small)`: khởi tạo với số chỗ mỗi loại.
- `addCar(carType) -> bool`: xe loại `carType` (`1` = big, `2` = medium, `3` = small) muốn vào. Xe **chỉ được đỗ đúng loại chỗ của nó**. Nếu còn chỗ: đỗ xe (giảm 1 chỗ) và trả `True`; hết chỗ: trả `False`.

**Ví dụ:**

```text
Input:  ["ParkingSystem", "addCar", "addCar", "addCar", "addCar"]
        [[1, 1, 0],       [1],      [2],      [3],      [1]]
Output: [null,            true,     true,     false,    false]
```

| Lệnh | Trước (big, medium, small) | Kết quả | Sau |
|---|---|:-:|---|
| `ParkingSystem(1,1,0)` | — | `null` | (1, 1, 0) |
| `addCar(1)` | (1, 1, 0) | `true` | (0, 1, 0) |
| `addCar(2)` | (0, 1, 0) | `true` | (0, 0, 0) |
| `addCar(3)` | (0, 0, 0) | `false` | (0, 0, 0) |
| `addCar(1)` | (0, 0, 0) | `false` | (0, 0, 0) |

**Constraints:** `0 <= big, medium, small <= 1000`; `carType ∈ {1, 2, 3}`; tối đa `1000` lần gọi `addCar`.

## 2. Nền tảng Python: class, `__init__`, `self`

### 2.1. Class là "khuôn", object là "sản phẩm"

- **Class** mô tả dữ liệu (thuộc tính) và hành vi (method).
- **Object** (đối tượng, instance) được tạo từ class: `obj = ClassName(...)`.
- `__init__` là method **khởi tạo**, tự chạy khi tạo object.
- `self` là chính object đang được dùng; `self.x = ...` gắn thuộc tính `x` vào object.

In [1]:
class Counter:
    def __init__(self, start):
        self.value = start          # thuộc tính gắn với object

    def inc(self):
        self.value += 1
        return self.value

c1 = Counter(10)
c2 = Counter(0)
print(c1.inc(), c1.inc(), "| c1.value =", c1.value)
print(c2.inc(), "| c2.value =", c2.value, "<- mỗi object có trạng thái riêng")

11 12 | c1.value = 12
1 | c2.value = 1 <- mỗi object có trạng thái riêng


### 2.2. Gọi method: `obj.method(arg)` = `ClassName.method(obj, arg)`

Python tự truyền object vào tham số đầu tiên `self`. Đó là lý do mọi method trong class đều có `self` đứng đầu.

In [2]:
c = Counter(5)
print(c.inc())             # cách gọi thường
print(Counter.inc(c))      # cách gọi "lộ" self — cùng kết quả

6
7


### 2.3. Quên `self.` — lỗi kinh điển

Viết `value = start` (không có `self.`) chỉ tạo **biến cục bộ** trong `__init__`, biến mất khi hàm kết thúc. Method khác sẽ không thấy nó.

In [3]:
class Broken:
    def __init__(self, start):
        value = start              # biến cục bộ, KHÔNG phải thuộc tính

    def get(self):
        return self.value

try:
    Broken(3).get()
except AttributeError as e:
    print("AttributeError:", e)

AttributeError: 'Broken' object has no attribute 'value'


### 2.4. LeetCode chạy bài design như thế nào?

LeetCode đưa 2 mảng: `ops` (tên lệnh) và `args` (tham số). Lệnh đầu là tên class → tạo object, các lệnh sau gọi method tương ứng. Kết quả của constructor in ra `null` (trong Python là `None`).

Ta viết hàm `run_ops` mô phỏng y hệt, dùng:
- `getattr(obj, "addCar")` — lấy method theo **tên dạng chuỗi**.
- `f(*args)` — dấu `*` **trải** list tham số thành các đối số riêng: `f(*[1, 1, 0])` = `f(1, 1, 0)`.

In [4]:
def add3(a, b, c):
    return a + b + c
params = [1, 2, 3]
print(add3(*params))
print(getattr("hello", "upper")())   # gọi "hello".upper() qua tên chuỗi

def run_ops(cls, ops, args):
    obj = cls(*args[0])
    out = [None]
    for name, a in zip(ops[1:], args[1:]):
        out.append(getattr(obj, name)(*a))
    return out

6
HELLO


## 3. Cách 1 — Ba thuộc tính riêng + `if/elif` (trực giác nhất)

Lưu số chỗ **còn trống** của mỗi loại vào 3 thuộc tính `self.big`, `self.medium`, `self.small`. Khi `addCar`:
1. Chọn đúng loại theo `carType`.
2. Nếu còn chỗ (`> 0`): giảm 1, trả `True`.
3. Ngược lại trả `False`.

In [5]:
class ParkingSystemIf:
    def __init__(self, big: int, medium: int, small: int):
        self.big = big
        self.medium = medium
        self.small = small

    def addCar(self, carType: int) -> bool:
        if carType == 1:
            if self.big > 0:
                self.big -= 1
                return True
            return False
        elif carType == 2:
            if self.medium > 0:
                self.medium -= 1
                return True
            return False
        else:
            if self.small > 0:
                self.small -= 1
                return True
            return False

ops = ["ParkingSystem", "addCar", "addCar", "addCar", "addCar"]
args = [[1, 1, 0], [1], [2], [3], [1]]
print(run_ops(ParkingSystemIf, ops, args))   # [None, True, True, False, False]

[None, True, True, False, False]


## 4. Nút thắt của cách 1

Code đúng, nhưng **ba khối gần như y hệt nhau** — chỉ khác tên thuộc tính. Hậu quả:
- Dễ sửa một khối mà quên khối khác (copy-paste bug).
- Nếu bãi có 10 loại chỗ, phải viết 10 nhánh.

Câu hỏi: `carType` đã là **số 1, 2, 3** — có thể dùng nó làm **chỉ số** để chọn ô nhớ, thay vì `if/elif`?

## 5. Insight — Dùng `carType` làm chỉ số list

Lưu `self.slots = [big, medium, small]`. Loại `carType` nằm ở chỉ số `carType - 1` (vì list bắt đầu từ 0).

Hoặc khéo hơn: thêm một phần tử "đệm" ở đầu `self.slots = [0, big, medium, small]` để `self.slots[carType]` dùng trực tiếp, không cần `- 1`.

In [6]:
big, medium, small = 1, 1, 0
slots = [big, medium, small]
for car_type in [1, 2, 3]:
    print(f"carType={car_type} -> chỉ số {car_type - 1} -> còn {slots[car_type - 1]} chỗ")

padded = [0, big, medium, small]
print("Có đệm:", padded, "| padded[2] =", padded[2], "(medium)")

carType=1 -> chỉ số 0 -> còn 1 chỗ
carType=2 -> chỉ số 1 -> còn 1 chỗ
carType=3 -> chỉ số 2 -> còn 0 chỗ
Có đệm: [0, 1, 1, 0] | padded[2] = 1 (medium)


## 6. Lời giải chính — list theo chỉ số (giảng từng đoạn)

### 6.1. `__init__`: lưu số chỗ còn trống

In [7]:
class ParkingSystemList:
    def __init__(self, big: int, medium: int, small: int):
        self.slots = [0, big, medium, small]   # chỉ số 0 bỏ trống cho tiện

    def addCar(self, carType: int) -> bool:
        if self.slots[carType] == 0:           # hết chỗ loại này
            return False
        self.slots[carType] -= 1               # đỗ xe: chiếm 1 chỗ
        return True

p = ParkingSystemList(1, 1, 0)
print("ban đầu:", p.slots)

ban đầu: [0, 1, 1, 0]


### 6.2. `addCar`: kiểm tra → cập nhật → trả kết quả

Thứ tự quan trọng: **kiểm tra trước, giảm sau**. Nếu giảm trước rồi mới kiểm tra, số chỗ có thể âm.

In [8]:
for t in [1, 2, 3, 1]:
    before = list(p.slots)          # sao chép để in "trước"
    res = p.addCar(t)
    print(f"addCar({t}) | trước={before} -> {res} | sau={p.slots}")

addCar(1) | trước=[0, 1, 1, 0] -> True | sau=[0, 0, 1, 0]
addCar(2) | trước=[0, 0, 1, 0] -> True | sau=[0, 0, 0, 0]
addCar(3) | trước=[0, 0, 0, 0] -> False | sau=[0, 0, 0, 0]
addCar(1) | trước=[0, 0, 0, 0] -> False | sau=[0, 0, 0, 0]


Ở trên dùng `list(p.slots)` để **sao chép**. Nếu viết `before = p.slots`, `before` chỉ là **tên khác** của cùng một list, nên khi `p.slots` đổi thì `before` cũng đổi theo.

In [9]:
a = [1, 2]
alias = a
copy = list(a)
a[0] = 99
print("alias:", alias, "| copy:", copy)

alias: [99, 2] | copy: [1, 2]


## 7. Cách 3 — `dict` (dễ đọc khi loại không phải số liên tiếp)

Nếu loại xe là chuỗi (`"big"`, `"medium"`...) hoặc số không liên tiếp, dùng `dict` ánh xạ `loại → số chỗ còn`.

In [10]:
class ParkingSystemDict:
    def __init__(self, big: int, medium: int, small: int):
        self.free = {1: big, 2: medium, 3: small}

    def addCar(self, carType: int) -> bool:
        if self.free[carType] > 0:
            self.free[carType] -= 1
            return True
        return False

print(run_ops(ParkingSystemDict, ops, args))

[None, True, True, False, False]


### 7.1. Biến thể "rút gọn": giảm trước rồi so sánh?

Một số người viết `addCar` cực ngắn bằng cách **trả về kết quả so sánh** sau khi giảm. Ví dụ: giảm luôn rồi trả `self.slots[t] >= 0`. Cách này đúng **về giá trị trả về** nhưng làm số chỗ âm dần (trạng thái "bẩn"), khó mở rộng (ví dụ thêm `removeCar`). Ta xem để biết, không khuyến khích.

In [11]:
class ParkingSystemShort:
    def __init__(self, big, medium, small):
        self.slots = [0, big, medium, small]

    def addCar(self, carType):
        self.slots[carType] -= 1
        return self.slots[carType] >= 0

p = ParkingSystemShort(1, 1, 0)
print([p.addCar(t) for t in [1, 2, 3, 1]], "| slots bị âm:", p.slots)

[True, True, False, False] | slots bị âm: [0, -1, 0, -1]


Ở cell trên dùng **list comprehension có tác dụng phụ** `[p.addCar(t) for t in ...]`: mỗi phần tử gọi `addCar` theo thứ tự trái → phải, nên thứ tự lệnh được giữ nguyên.

## 8. TRACE từng bước

Tạo `ParkingSystem(2, 0, 1)` rồi gọi `addCar` theo dãy `[1, 3, 1, 1, 2, 3]`:

| Lệnh | slots trước `[_, B, M, S]` | Kiểm tra | Kết quả | slots sau |
|---|---|---|:-:|---|
| `addCar(1)` | `[0, 2, 0, 1]` | B=2 > 0 | `True` | `[0, 1, 0, 1]` |
| `addCar(3)` | `[0, 1, 0, 1]` | S=1 > 0 | `True` | `[0, 1, 0, 0]` |
| `addCar(1)` | `[0, 1, 0, 0]` | B=1 > 0 | `True` | `[0, 0, 0, 0]` |
| `addCar(1)` | `[0, 0, 0, 0]` | B=0 | `False` | không đổi |
| `addCar(2)` | `[0, 0, 0, 0]` | M=0 (từ đầu) | `False` | không đổi |
| `addCar(3)` | `[0, 0, 0, 0]` | S=0 | `False` | không đổi |

In [12]:
NAMES = {1: "big", 2: "medium", 3: "small"}

def trace_parking(big, medium, small, cars):
    p = ParkingSystemList(big, medium, small)
    print(f"ParkingSystem({big}, {medium}, {small}) -> slots = {p.slots}")
    results = []
    for t in cars:
        before = p.slots[t]
        res = p.addCar(t)
        results.append(res)
        print(f"  addCar({t}={NAMES[t]:<6}) còn {before} chỗ -> {'True ' if res else 'False'} | slots = {p.slots}")
    return results

print(trace_parking(2, 0, 1, [1, 3, 1, 1, 2, 3]))

ParkingSystem(2, 0, 1) -> slots = [0, 2, 0, 1]
  addCar(1=big   ) còn 2 chỗ -> True  | slots = [0, 1, 0, 1]
  addCar(3=small ) còn 1 chỗ -> True  | slots = [0, 1, 0, 0]
  addCar(1=big   ) còn 1 chỗ -> True  | slots = [0, 0, 0, 0]
  addCar(1=big   ) còn 0 chỗ -> False | slots = [0, 0, 0, 0]
  addCar(2=medium) còn 0 chỗ -> False | slots = [0, 0, 0, 0]
  addCar(3=small ) còn 0 chỗ -> False | slots = [0, 0, 0, 0]
[True, True, True, False, False, False]


Trace ví dụ của đề:

In [13]:
print(trace_parking(1, 1, 0, [1, 2, 3, 1]))

ParkingSystem(1, 1, 0) -> slots = [0, 1, 1, 0]
  addCar(1=big   ) còn 1 chỗ -> True  | slots = [0, 0, 1, 0]
  addCar(2=medium) còn 1 chỗ -> True  | slots = [0, 0, 0, 0]
  addCar(3=small ) còn 0 chỗ -> False | slots = [0, 0, 0, 0]
  addCar(1=big   ) còn 0 chỗ -> False | slots = [0, 0, 0, 0]
[True, True, False, False]


## 9. Code nộp LeetCode

### 9.1. Cách chính — list theo chỉ số

In [14]:
class ParkingSystem:

    def __init__(self, big: int, medium: int, small: int):
        self.slots = [0, big, medium, small]

    def addCar(self, carType: int) -> bool:
        if self.slots[carType] == 0:
            return False
        self.slots[carType] -= 1
        return True


# Your ParkingSystem object will be instantiated and called as such:
obj = ParkingSystem(1, 1, 0)
param_1 = obj.addCar(1)
print(param_1)
print(run_ops(ParkingSystem, ops, args))

True
[None, True, True, False, False]


### 9.2. Cách ba thuộc tính (đã viết ở mục 3 — `ParkingSystemIf`), cách `dict` (`ParkingSystemDict`)

Khi nộp LeetCode chỉ cần **đổi tên class** thành `ParkingSystem`. Dưới đây là bản `dict` gọn để nộp:

In [15]:
class ParkingSystemSubmitDict:
    def __init__(self, big: int, medium: int, small: int):
        self.free = {1: big, 2: medium, 3: small}

    def addCar(self, carType: int) -> bool:
        if self.free[carType]:
            self.free[carType] -= 1
            return True
        return False

print(run_ops(ParkingSystemSubmitDict, ops, args))

[None, True, True, False, False]


`if self.free[carType]:` — số `0` được coi là `False`, số khác 0 là `True` (truthiness), nên tương đương `> 0` khi số chỗ không âm.

## 10. Vì sao đúng? (invariant)

**Invariant:** sau mỗi lệnh, `slots[t]` = (số chỗ ban đầu của loại `t`) − (số xe loại `t` đã được nhận) và luôn `>= 0`.

- Khởi tạo: chưa nhận xe nào, `slots[t]` = số chỗ ban đầu ✓.
- `addCar(t)` khi `slots[t] > 0`: còn ít nhất một chỗ đúng loại → nhận xe, giảm 1, vẫn `>= 0` ✓.
- `addCar(t)` khi `slots[t] == 0`: mọi chỗ loại `t` đã có xe → từ chối, không đổi ✓.
- Các loại khác không bị đụng tới → xe chỉ đỗ đúng loại ✓.

### 10.1. Độ phức tạp

| Cách | `__init__` | `addCar` | Space |
|---|---|---|---|
| 1. Ba thuộc tính + `if/elif` | `O(1)` | `O(1)` | `O(1)` |
| 2. List theo chỉ số | `O(1)` | `O(1)` | `O(1)` |
| 3. `dict` | `O(1)` | `O(1)` trung bình | `O(1)` (3 key) |

Với `q` lần gọi, tổng thời gian `O(q)`. Ta **không** lưu từng chỗ đỗ (list 1000 phần tử) — chỉ cần **đếm**, vì đề không hỏi xe nào đỗ ở đâu.

## 11. Bộ test

1. Ví dụ đề.
2. Biên: mọi loại 0 chỗ; mỗi loại 1000 chỗ và gọi 1000 lần.
3. Xe không được đỗ chỗ khác loại dù chỗ khác còn trống.
4. Đối chiếu các cài đặt trên dãy lệnh ngẫu nhiên với mô hình tham chiếu viết kiểu "danh sách chỗ".

In [16]:
impls = [ParkingSystemIf, ParkingSystemList, ParkingSystemDict, ParkingSystem, ParkingSystemSubmitDict]

def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<36} PASS")

for cls in impls:
    assert run_ops(cls, ops, args) == [None, True, True, False, False]
check("ví dụ đề (5 cài đặt)", run_ops(ParkingSystem, ops, args), [None, True, True, False, False])

p = ParkingSystem(0, 0, 0)
check("bãi không có chỗ", [p.addCar(t) for t in [1, 2, 3]], [False, False, False])

p = ParkingSystem(0, 5, 0)
check("không đỗ chỗ khác loại", [p.addCar(1), p.addCar(3), p.addCar(2)], [False, False, True])

p = ParkingSystem(1000, 1000, 1000)
res = [p.addCar(1) for _ in range(1000)]
check("1000 xe big vừa đủ 1000 chỗ", all(res) and p.addCar(1) is False, True)

ví dụ đề (5 cài đặt)                 PASS
bãi không có chỗ                     PASS
không đỗ chỗ khác loại               PASS
1000 xe big vừa đủ 1000 chỗ          PASS


In [17]:
import random

class ReferenceParking:
    # Mô hình tham chiếu: lưu từng chỗ đỗ dạng list True/False (chậm nhưng hiển nhiên đúng)
    def __init__(self, big, medium, small):
        self.spots = {1: [False] * big, 2: [False] * medium, 3: [False] * small}

    def addCar(self, carType):
        for k, taken in enumerate(self.spots[carType]):
            if not taken:
                self.spots[carType][k] = True
                return True
        return False

rng = random.Random(1603)
for _ in range(300):
    caps = [rng.randint(0, 4) for _ in range(3)]
    cars = [rng.randint(1, 3) for _ in range(rng.randint(1, 20))]
    ref = ReferenceParking(*caps)
    expected = [ref.addCar(t) for t in cars]
    for cls in impls:
        obj = cls(*caps)
        assert [obj.addCar(t) for t in cars] == expected
print("300 kịch bản ngẫu nhiên: 5 cài đặt khớp mô hình tham chiếu")

300 kịch bản ngẫu nhiên: 5 cài đặt khớp mô hình tham chiếu


## 12. Lỗi người mới hay gặp

### Lỗi 1: Giảm trước, kiểm tra sau → số chỗ âm và kết quả sai

In [18]:
class WrongOrder:
    def __init__(self, big, medium, small):
        self.slots = [0, big, medium, small]

    def addCar(self, carType):
        self.slots[carType] -= 1
        return self.slots[carType] > 0          # sai: chỗ cuối cùng bị từ chối

w = WrongOrder(1, 1, 0)
print("sai:", [w.addCar(t) for t in [1, 2, 3, 1]], "| đúng: [True, True, False, False]")

sai: [False, False, False, False] | đúng: [True, True, False, False]


### Lỗi 2: Lệch chỉ số — dùng `[big, medium, small]` nhưng truy cập `slots[carType]`

`carType = 3` → `slots[3]` vượt biên (`IndexError`); `carType = 1` lại trỏ vào `medium`.

In [19]:
class WrongIndex:
    def __init__(self, big, medium, small):
        self.slots = [big, medium, small]

    def addCar(self, carType):
        if self.slots[carType] > 0:            # thiếu "- 1"
            self.slots[carType] -= 1
            return True
        return False

w = WrongIndex(1, 0, 5)
print("addCar(1) ->", w.addCar(1), "(lẽ ra True vì big còn 1 chỗ)")
try:
    w.addCar(3)
except IndexError as e:
    print("addCar(3) -> IndexError:", e)

addCar(1) -> False (lẽ ra True vì big còn 1 chỗ)
addCar(3) -> IndexError: list index out of range


### Lỗi 3: Dùng biến của class (class attribute) thay vì của object

Khai báo `slots = [...]` ở **thân class** (ngoài `__init__`) thì list đó được **chia sẻ** giữa mọi object — bãi này đỗ xe làm bãi kia mất chỗ!

In [20]:
class WrongShared:
    slots = [0, 1, 1, 1]          # thuộc về CLASS, mọi object dùng chung

    def __init__(self, big, medium, small):
        pass                      # quên gán self.slots riêng

    def addCar(self, carType):
        if self.slots[carType] > 0:
            self.slots[carType] -= 1
            return True
        return False

a, b2 = WrongShared(1, 1, 1), WrongShared(1, 1, 1)
print("bãi a:", a.addCar(1), "| bãi b:", b2.addCar(1), "<- b bị ảnh hưởng bởi a")

bãi a: True | bãi b: False <- b bị ảnh hưởng bởi a


### Lỗi 4: Trả chuỗi `"true"` / `"false"` thay vì `True` / `False`

Output LeetCode hiển thị `true` (chữ thường) vì đó là cú pháp JSON/Java. Trong Python phải trả **bool** `True`/`False`.

In [21]:
print(type(True), type("true"), True == "true")

<class 'bool'> <class 'str'> False


## 13. Mẹo nhận dạng pattern

- Bài **Design** = "class + trạng thái + method". Hỏi: *trạng thái tối thiểu nào đủ để trả lời mọi method?* (ở đây chỉ cần **3 con số**).
- Loại là số nhỏ liên tiếp → **list theo chỉ số** (có thể đệm phần tử 0). Loại là chuỗi/thưa → **dict**.
- Mọi method thay đổi trạng thái: **kiểm tra điều kiện trước, cập nhật sau**.
- Bài liên quan: 705 Design HashSet, 706 Design HashMap, 1396 Design Underground System, 2043 Simple Bank System, 1845 Seat Reservation Manager, 155 Min Stack.

## 14. Biến thể

### 15.1. Thêm `removeCar(carType)` — xe rời bãi

In [22]:
class ParkingWithRemove:
    def __init__(self, big, medium, small):
        self.capacity = [0, big, medium, small]
        self.slots = [0, big, medium, small]

    def addCar(self, carType):
        if self.slots[carType] == 0:
            return False
        self.slots[carType] -= 1
        return True

    def removeCar(self, carType):
        if self.slots[carType] == self.capacity[carType]:   # không có xe loại này
            return False
        self.slots[carType] += 1
        return True

p = ParkingWithRemove(1, 0, 0)
print(p.addCar(1), p.addCar(1), p.removeCar(1), p.addCar(1), p.removeCar(2))

True False True True False


### 15.2. Xe nhỏ được đỗ chỗ lớn hơn nếu chỗ đúng loại đã hết

Xe loại `t` thử lần lượt các chỗ `t, t-1, ..., 1` (từ vừa khít tới rộng nhất).

In [23]:
class ParkingFlexible:
    def __init__(self, big, medium, small):
        self.slots = [0, big, medium, small]

    def addCar(self, carType):
        for t in range(carType, 0, -1):        # range đi lùi: carType, ..., 1
            if self.slots[t] > 0:
                self.slots[t] -= 1
                return True
        return False

p = ParkingFlexible(1, 0, 1)
print([p.addCar(t) for t in [3, 3, 3, 1]], p.slots)   # xe nhỏ thứ 2 đỗ chỗ big

[True, True, False, False] [0, 0, 0, 0]


`range(carType, 0, -1)` sinh dãy **giảm dần** từ `carType` xuống `1` (dừng trước `0`).

### 15.3. Thống kê: số xe mỗi loại đang trong bãi

In [24]:
def occupancy(p: ParkingSystem, big, medium, small):
    caps = [0, big, medium, small]
    return {NAMES[t]: caps[t] - p.slots[t] for t in (1, 2, 3)}

p = ParkingSystem(3, 2, 1)
for t in [1, 1, 2, 3, 3]:
    p.addCar(t)
print(occupancy(p, 3, 2, 1))

{'big': 2, 'medium': 1, 'small': 1}


## 15. Bài tập tự luyện (TODO)

**Bài 1 — tính tay.** `ParkingSystem(0, 2, 1)` rồi `addCar` theo `[2, 3, 2, 2, 3, 1]`. Viết list kết quả.

**Bài 2 — đoán output.** Sau `a = [0, 1, 2]; b = a; b[0] = 5`, `a` là gì? Còn nếu `b = list(a)`?

### Bài 3 — Điền TODO: cài đặt bằng list theo chỉ số

In [25]:
class ExerciseParking:
    def __init__(self, big: int, medium: int, small: int):
        # TODO: self.slots = [...]
        pass

    def addCar(self, carType: int) -> bool:
        # TODO: kiểm tra còn chỗ -> giảm -> True; hết chỗ -> False
        pass

# Test sau khi làm xong:
# assert run_ops(ExerciseParking, ops, args) == [None, True, True, False, False]

### Bài 4 — Tự viết `removeCar`

In [26]:
class ExerciseParkingRemove:
    def __init__(self, big, medium, small):
        # TODO: lưu cả capacity và slots
        pass

    def addCar(self, carType):
        # TODO
        pass

    def removeCar(self, carType):
        # TODO: chỉ trả True nếu thật sự có xe loại đó trong bãi
        pass

# p = ExerciseParkingRemove(1, 0, 0)
# assert [p.addCar(1), p.removeCar(1), p.removeCar(1)] == [True, True, False]

### Bài 5 — Biến thể: bãi với số loại chỗ tùy ý

`GeneralParking(capacities: list[int])`, loại xe từ `1` tới `len(capacities)`.

In [27]:
class ExerciseGeneralParking:
    def __init__(self, capacities: list[int]):
        # TODO
        pass

    def addCar(self, carType: int) -> bool:
        # TODO
        pass

# g = ExerciseGeneralParking([1, 0, 2, 1])
# assert [g.addCar(t) for t in [4, 4, 3, 2, 3, 3]] == [True, False, True, False, True, False]

### Bài 6 — Mini challenge: chỗ đỗ linh hoạt + trả về **loại chỗ** đã dùng

`addCar(carType)` trả về số loại chỗ xe được đỗ (1/2/3), hoặc `-1` nếu không còn chỗ phù hợp. Xe ưu tiên chỗ vừa khít nhất.

In [28]:
class ExerciseFlexible:
    def __init__(self, big, medium, small):
        # TODO
        pass

    def addCar(self, carType) -> int:
        # TODO
        pass

# f = ExerciseFlexible(1, 1, 0)
# assert [f.addCar(3), f.addCar(3), f.addCar(3)] == [2, 1, -1]

## 16. Gợi ý 3 mức

**Bài 1:** M1: medium 2, small 1, big 0. M2: hai xe medium đầu True, cái thứ ba False. M3: xe small thứ hai False, xe big False.

**Bài 2:** M1: `b = a` không sao chép. M2: `list(a)` tạo list mới. M3: `[5, 1, 2]` và `[0, 1, 2]`.

**Bài 3:** M1: đệm 0 ở đầu. M2: `if self.slots[carType] == 0: return False`. M3: `self.slots[carType] -= 1; return True`.

**Bài 4:** M1: cần biết sức chứa ban đầu. M2: "có xe" ⟺ `slots < capacity`. M3: `removeCar` tăng `slots` lên 1.

**Bài 5:** M1: `self.slots = [0] + capacities`. M2: giống lời giải chính. M3: không cần `if/elif` nào.

**Bài 6:** M1: duyệt `range(carType, 0, -1)`. M2: trả về `t` khi tìm được chỗ. M3: hết vòng lặp → `-1`.

## 17. Đáp án tham khảo

**Bài 1:** `[True, True, True, False, False, False]`.

**Bài 2:** `a = [5, 1, 2]` (vì `b` là cùng một list); với `b = list(a)` thì `a` vẫn `[0, 1, 2]`.

In [29]:
class AnswerParking:
    def __init__(self, big, medium, small):
        self.slots = [0, big, medium, small]

    def addCar(self, carType):
        if self.slots[carType] == 0:
            return False
        self.slots[carType] -= 1
        return True

class AnswerParkingRemove(AnswerParking):       # kế thừa: dùng lại addCar
    def __init__(self, big, medium, small):
        super().__init__(big, medium, small)
        self.capacity = [0, big, medium, small]

    def removeCar(self, carType):
        if self.slots[carType] == self.capacity[carType]:
            return False
        self.slots[carType] += 1
        return True

class AnswerGeneralParking:
    def __init__(self, capacities):
        self.slots = [0] + list(capacities)

    def addCar(self, carType):
        if self.slots[carType] == 0:
            return False
        self.slots[carType] -= 1
        return True

class AnswerFlexible:
    def __init__(self, big, medium, small):
        self.slots = [0, big, medium, small]

    def addCar(self, carType):
        for t in range(carType, 0, -1):
            if self.slots[t] > 0:
                self.slots[t] -= 1
                return t
        return -1

p = AnswerParking(0, 2, 1)
assert [p.addCar(t) for t in [2, 3, 2, 2, 3, 1]] == [True, True, True, False, False, False]
a = [0, 1, 2]; bb = a; bb[0] = 5
assert a == [5, 1, 2]
a = [0, 1, 2]; bb = list(a); bb[0] = 5
assert a == [0, 1, 2]
assert run_ops(AnswerParking, ops, args) == [None, True, True, False, False]
p = AnswerParkingRemove(1, 0, 0)
assert [p.addCar(1), p.removeCar(1), p.removeCar(1)] == [True, True, False]
g = AnswerGeneralParking([1, 0, 2, 1])
assert [g.addCar(t) for t in [4, 4, 3, 2, 3, 3]] == [True, False, True, False, True, False]
f = AnswerFlexible(1, 1, 0)
assert [f.addCar(3), f.addCar(3), f.addCar(3)] == [2, 1, -1]
print("Đáp án tham khảo: PASS toàn bộ!")

Đáp án tham khảo: PASS toàn bộ!


Trong đáp án Bài 4 dùng **kế thừa** (`class Con(Cha)`) và `super().__init__(...)` để gọi `__init__` của lớp cha — tránh viết lại `addCar`. Đây là kiến thức thêm, không bắt buộc cho bài gốc.

## 18. Template nhớ nhanh + checklist submit

```python
class ParkingSystem:

    def __init__(self, big: int, medium: int, small: int):
        self.slots = [0, big, medium, small]   # đệm chỉ số 0 -> slots[carType]

    def addCar(self, carType: int) -> bool:
        if self.slots[carType] == 0:           # kiểm tra trước
            return False
        self.slots[carType] -= 1               # cập nhật sau
        return True
```

**Câu thần chú:** *"Trạng thái tối thiểu = 3 con số; kiểm tra rồi mới trừ."*

**Checklist trước khi submit:**
- [ ] Tên class `ParkingSystem`, method `addCar` đúng chữ hoa/thường?
- [ ] Mọi thuộc tính gán qua `self.` trong `__init__` (không dùng biến của class)?
- [ ] Chỉ số đúng (`carType` với list có đệm, hoặc `carType - 1` không đệm)?
- [ ] Kiểm tra còn chỗ **trước** khi giảm; số chỗ không bao giờ âm?
- [ ] Trả `True`/`False` kiểu bool?